In [1]:
import os
from dotenv import load_dotenv

# LangChain & LangGraph Imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import shutil
from langchain_core.documents import Document
import re

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("🚨 OPENAI_API_KEY not found! Check your .env file.")

# Enable LangSmith Tracing (Highly recommended for Agentic RAG)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"] = "Nexteir_Second_Brain_Lab"

print("✅ Environment loaded. Tracing enabled.")

✅ Environment loaded. Tracing enabled.


In [2]:
# 1. Extract
pdf_path = "../data/NEXTIER_Internship_Bugs.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"📄 Loaded {len(docs)} pages from the internship logs.")

# 2. Merge all pages into a single string
raw_text = "\n".join([doc.page_content for doc in docs])

# [FIX] 3. Clean the text by stripping out page markers
clean_text = re.sub(r'---\s*PAGE\s*\d+\s*---', '\n', raw_text)

# [FIX] 3.1 Collapse excessive whitespace
clean_text = re.sub(r'\n\s*\n', '\n\n', clean_text)

# [NEW FIX] 3.2 Standardize all problem headers (e.g., "//PROBLEM 75:" -> "//problem75:")
# This ensures Rule 1 in app.py and the Isolation Rule always find a match.
clean_text = re.sub(r'//\s*problem\s*(\d+)\s*:', r'//problem\1:', clean_text, flags=re.IGNORECASE)

# Re-wrap into a LangChain Document
merged_docs = [Document(page_content=clean_text, metadata={"source": pdf_path})]

# 4. Transform: Split the cleaned text
text_splitter = RecursiveCharacterTextSplitter(
    separators=["//problem", "\n\n", "\n", " ", ""], 
    chunk_size=2500, 
    chunk_overlap=0,
    add_start_index=True 
)

splits = text_splitter.split_documents(merged_docs)
print(f"🧩 Split into {len(splits)} chunks.")

# --- QUICK DIAGNOSTIC ---
# Let's search the raw Python list BEFORE it even touches ChromaDB
problem_13_chunks = [chunk for chunk in splits if "//problem13:" in chunk.page_content]

print(f"🕵️ Found {len(problem_13_chunks)} chunks containing '//problem13:' in raw memory.")

if problem_13_chunks:
    print("\n📄 Preview of the first match:")
    print(problem_13_chunks[0].page_content[:300])
else:
    print("🚨 FATAL: Problem 13 was completely erased during the regex cleaning or text splitting phase!")

📄 Loaded 227 pages from the internship logs.
🧩 Split into 188 chunks.
🕵️ Found 1 chunks containing '//problem13:' in raw memory.

📄 Preview of the first match:
//problem13:  
///OTP  VALUE  HIDDEN  UI  ISSUE

in

src/screens/founder/auth/authStyles.ts  The  dark-mode  screenshot  you  shared  clearly  shows  the  6th  input  box  completely  cut  off  and  the  5th  box  
partially

hidden.

This

is

a

classic

"overflow"

layout

error.
 Here  is  the  


In [3]:
# 3. Load: Create embeddings and store in ChromaDB
persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

import shutil
import os

# Robust cleanup: ensuring every 'Run All' starts with a fresh index
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
    print("🗑️ Cleared existing Chroma database for a clean rebuild.")

# This creates the physical database files in backend/chroma_db
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    persist_directory=persist_directory
)

# [Verification Test] 
# We test the actual "trouble spot" directly in the notebook
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
res = retriever.invoke("//problem13:")

if len(res) > 0:
    print(f"✅ Verified: Found {len(res)} chunks for Problem 13.")
    print(f"📄 Preview: {res[0].page_content[:150]}...")
else:
    print("❌ Failure: Problem 13 not found in the vector store.")

🗑️ Cleared existing Chroma database for a clean rebuild.
✅ Verified: Found 2 chunks for Problem 13.
📄 Preview: //problem39:  
Error  Summary  
Root  Cause:  A  Nested  Touchable  Conflict .  The  userInfo container  in  PostCard.tsx was  a  TouchableOpacity wit...


In [4]:
@tool
def search_internship_history(query: str) -> str:
    """
    Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
    containing debugging fixes (//problemXX), architectural decisions, 
    and useful CLI/Terminal commands (//useful command).
    """
    # 1. Multi-Query Expansion: Let the LLM "think" of better search terms
    # This is what makes it "Lazy Search" friendly.
    expansion_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    search_variants = expansion_llm.invoke(
        f"Generate 3 diverse search queries to find information about '{query}' "
        "in technical internship logs. Focus on keywords, problem IDs, and file paths. "
        "Respond with only the queries, one per line."
    ).content.split("\n")

    # Add the original query to the list
    all_queries = [query] + [q.strip() for q in search_variants if q.strip()]
    
    # 2. Execute a broad search across all variants
    all_docs = []
    for q in all_queries:
        all_docs.extend(retriever.invoke(q))

    # 3. Deduplicate (so we don't show the same chunk twice)
    unique_docs = {doc.page_content: doc for doc in all_docs}.values()

    # 4. Format with the Strict Rules
    formatted_context = "\n\n---\n\n".join(
        [f"Page {doc.metadata.get('page', 'Unknown')}:\n{doc.page_content}" for doc in unique_docs]
    )
    
    return formatted_context

print("🛠️ Tool 'search_internship_history' defined successfully.")
print(f"Tool Description given to LLM: {search_internship_history.description}")

🛠️ Tool 'search_internship_history' defined successfully.
Tool Description given to LLM: Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
containing debugging fixes (//problemXX), architectural decisions, 
and useful CLI/Terminal commands (//useful command).


In [5]:
# Initialize the model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind the tool to the LLM
tools = [search_internship_history]
llm_with_tools = llm.bind_tools(tools)

# Ask a specific question that requires the tool
test_message = HumanMessage(content="I'm getting an Android layout collapse in my React Native app. How did we fix this during the internship?")
response = llm_with_tools.invoke([test_message])

# Check if the AI decided to use the tool
if response.tool_calls:
    print("🧠 The Agent decided to use a tool!")
    for tool_call in response.tool_calls:
        print(f"-> Tool Selected: {tool_call['name']}")
        print(f"-> Search Query Generated by AI: {tool_call['args']['query']}")
else:
    print("⚠️ The Agent answered directly without using the tool.")
    print(response.content)

🧠 The Agent decided to use a tool!
-> Tool Selected: search_internship_history
-> Search Query Generated by AI: Android layout collapse React Native
